# Tutorial_4_Retrieval_Analysis

This tutorial walks you through how to read, interpret, and visualize the retrieval results.

We will load retained MCMC samples, model configurations, and opacity data, then prepare them for post-retrieval analysis. This notebook uses the MCMC loader and parameterization; it does not load MultiNest outputs.

In [2]:
from pathlib import Path
import os
import sys

# Use one working directory for Brewster imports and repository data paths.
project_root = Path.cwd().resolve()
for candidate in (project_root, *project_root.parents):
    if (candidate / "utils.py").is_file() and (candidate / "retrieval_run.py").is_file():
        project_root = candidate
        break
else:
    raise FileNotFoundError("Start this notebook from within the Brewster repository.")

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Brewster repository:", project_root)

Brewster repository: /Users/fw23aao/spidernail/brewster_trail/v2_master/brewster_v2


In [3]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.image as mgimg
import matplotlib.colors as colors
import scipy as sp
import numpy as np
import emcee
import os 
from collections import namedtuple
import settings
import utils
import test_module
import retrieval_run
import corner
import pickle as pickle
import TPmod
from specops import proc_spec
from IPython.display import display
%matplotlib inline

## 1. Load the MCMC end chains, arguments and opacities

In [6]:
path = "/Users/fw23aao/spidernail/brewster_trail/v2_master/result/"

runname = "V2_G570D_test"

# OK finish? 1 for yes, 0 for no.
fin = 0
flatendchain, flatendprobs,ndim = utils.get_endchain(runname,fin,path)
if not np.all(np.isfinite(flatendprobs)):
    raise ValueError("Retained log-posterior values contain non-finite entries; inspect the chain before selecting a sample.")
map_sample_index = np.argmax(flatendprobs)
theta_map_sample = flatendchain[map_sample_index]
max_log_posterior_sample = flatendprobs[map_sample_index]
samples = flatendchain

argfile =path+runname+"_runargs.pic"
runargs = utils.pickle_load(argfile)
opacityfile =path+runname+"_opacities.pic"
opacities = utils.pickle_load(opacityfile)

# cloudfile =path+runname+"_cloudata.pic"
# cloudata = utils.pickle_load(cloudfile)

settings.init(runargs)
settings.linelist=opacities[0]
settings.cia=opacities[1]
# settings.cloudata=cloudata

settings.cloudata=runargs.cloudata

with open(path+runname+'_configs.pic', 'rb') as file:
    configs= pickle.load(file)
    
re_params=configs['re_params']
model_config_instance=configs['model_config']

## 2. Distinguish posterior selection, likelihood, and BIC

For MCMC, `flatendprobs` contains log prior + log likelihood. The legacy `get_endchain` console messages call these values "maximum likelihood", but they are log posterior. `theta_map_sample` is the highest-posterior retained sample, an approximation to the MAP rather than an optimized MAP solution. It is used for the parameter display, mass/radius calculation, and representative spectrum below.

The next calculation evaluates `test_module.lnlike` separately for every retained sample. This requires one forward-model evaluation per sample and can be expensive. `theta_ml_sample` is the highest-likelihood retained sample; it can differ from `theta_map_sample`. Neither selection establishes the continuous maximum likelihood.

BIC is defined using the maximum **data likelihood**, not the posterior. We report only a sample-based approximation, `BIC_sample`, which can overestimate BIC if the retained samples miss the likelihood maximum. The data count follows the likelihood's every-third-point selection for uniform FWHM, or all points for an R file. The parameter count excludes the P–T smoothing hyperparameter `gamma`, which enters the MCMC prior rather than the data likelihood; check for any other inactive parameters in custom configurations. BIC's usual regular-model and independent-data assumptions still need assessment before model comparison.

This calculation is restricted to MCMC: the MultiNest `lnlike` branch can include smoothing-prior terms, so its return value cannot automatically be treated as a pure data likelihood.

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # assuming you already have:
# # flatendchain, flatendprobs, ndim, all_params

# n_params = len(all_params)
# fig, axes = plt.subplots(n_params, 1, figsize=(10, 2*n_params), sharex=True)

# for i in range(n_params):
#     ax = axes[i]
#     ax.plot(flatendchain[5000:, i], lw=0.3, alpha=0.7)
#     ax.set_ylabel(all_params[i])
#     if i == 0:
#         ax.set_title("MCMC Chain Traces for Each Parameter")

# axes[-1].set_xlabel("Iteration")
# plt.tight_layout()
# plt.show()


In [ ]:
args_instance=settings.runargs
all_params,all_params_values =utils.get_all_parametres(re_params.dictionary)

params_master = namedtuple('params',all_params)
params_instance = params_master(*theta_map_sample)
params_instance

In [ ]:
if re_params.samplemode.lower() != "mcmc":
    raise ValueError("This likelihood/BIC calculation is for MCMC configurations only.")

sample_log_likelihood = np.empty(len(samples))
for i, theta in enumerate(samples):
    sample_log_likelihood[i] = test_module.lnlike(theta, re_params)
    if not np.isfinite(sample_log_likelihood[i]):
        raise ValueError(f"Non-finite data log likelihood at retained sample {i}.")

ml_sample_index = np.argmax(sample_log_likelihood)
theta_ml_sample = samples[ml_sample_index]
max_log_likelihood_sample = sample_log_likelihood[ml_sample_index]
log_likelihood_at_map_sample = sample_log_likelihood[map_sample_index]

n_data = runargs.obspec[0, ::3].size if runargs.fwhm is not None else runargs.obspec.shape[1]
k_likelihood = ndim - int(runargs.proftype in (1, 77))
BIC_sample = -2.0 * max_log_likelihood_sample + k_likelihood * np.log(n_data)

print("Highest retained log posterior:", max_log_posterior_sample)
print("Data log likelihood at highest-posterior sample:", log_likelihood_at_map_sample)
print("Highest retained data log likelihood:", max_log_likelihood_sample)
print("Posterior and likelihood select the same sample:", map_sample_index == ml_sample_index)
print("Sample-based BIC approximation:", BIC_sample)
print("Likelihood parameter count and data count:", k_likelihood, n_data)

In [ ]:
re_params.dictionary['pt']['params'].keys()

## 3. Extract the P–T Profile from the MCMC samples.
compute the median and credible intervals (1σ and 2σ) for the atmospheric temperature structure.

In [ ]:
all_params,all_params_values =utils.get_all_parametres(re_params.dictionary) 
params_master = namedtuple('params',all_params)
params_instance = params_master(*theta_map_sample)

intemp_keys = list(re_params.dictionary['pt']['params'].keys())

if runargs.proftype==1 or runargs.proftype==77:
    p_index_first=params_instance._fields.index(intemp_keys[1]) 
else:
    p_index_first=params_instance._fields.index(intemp_keys[0]) 
    
p_index_last=params_instance._fields.index(intemp_keys[-1])+1

Tsamples = samples[:,p_index_first:p_index_last]
nsamps = Tsamples.shape[0]
Tprofs = np.empty([64,Tsamples.shape[0]])
for i in range(0,nsamps):
    Tprofs[:,i] = TPmod.set_prof(runargs.proftype,runargs.coarsePress,runargs.press,Tsamples[i,:])
    
Tlays = np.empty([64,5])
for i in range(0,64):
    junk = Tprofs[i,:]
    junk2 = np.percentile(junk, [2.4,16, 50, 84,97.6],axis=0)
    junk3 = np.array(junk2)
    Tlays[i,:] = junk3[:]

In [ ]:
plt.rc('font',family='Times New Roman')
fig=plt.figure(dpi=150)
plt.axis([0., 4000.,3.0,-5.0])

logP = np.log10(runargs.press)

d1, = plt.plot(Tlays[:,2],logP,'k-',label=runname)
plt.fill_betweenx(logP,Tlays[:,1], Tlays[:,3], facecolor='red', alpha=0.3)
plt.fill_betweenx(logP,Tlays[:,0], Tlays[:,4], facecolor='red', alpha=0.1)

# Here are some condensation curves
enst = 10.0**4/(6.26 - 0.35*logP-0.70*0.0)
fost = 10.0**4/(5.89 - 0.37*logP-0.73*0.0)
iron = 10.0**4/(5.44 - 0.48*logP-0.48*0.0)
cr =  10.0**4/(6.528 - 0.491*logP-0.491*0.0)
al2o3 = 10.0**4 / (5.0139 - 0.21794*(logP) + 2.2636E-03*(logP)**2.0 - 0.580*0.0)
c1, = plt.plot(enst,logP,'--',color='blue',linewidth=1.5, label='MgSiO$_3$')
c2, = plt.plot(fost,logP,'--',color='pink',linewidth=1.5,label='Mg$_2$SiO$_4$')
c3, = plt.plot(iron,logP,'--',color='orange',linewidth=1.5, label='Fe')
c4, = plt.plot(cr,logP,'--',color='purple',linewidth=1.5, label='Cr')
c5, = plt.plot(al2o3,logP,'--',color='red',linewidth=1.5, label='Al$_2$O$_3$')

plt.legend(handles=[d1,c4,c1,c2,c3,c5])
plt.ylabel(r'log(P / bar)')
plt.xlabel('T / K')


## 4. Check the Mass and Radius from the retrieved parameters r2d2 and logg.

In [ ]:
D = 3.086e+16 * model_config_instance.dist

r2d2_index=params_instance._fields.index('r2d2')
r2d2 =theta_map_sample[r2d2_index]


logg_index=params_instance._fields.index('logg')
logg =theta_map_sample[logg_index]


if (r2d2 > 0.):
    R = np.sqrt(r2d2) * D
g = (10.**logg)/100.
M = (R**2 * g/(6.67E-11))/1.898E27
Rj = R / 69911.e3
print(M,Rj)

## 5. Corner Plot
Visualize the posterior distributions of all retrieved parameters using a corner plot.

In [ ]:
fig = corner.corner(samples,scale_hist=False,plot_datapoints =False,\
                    labels=all_params,\
                    quantiles=[0.16, 0.5, 0.84],show_titles=True, title_kwargs={"fontsize": 20},\
                    label_kwargs={"fontsize": 20})

## 6. Plot the Fitted Spectrum
Generate the spectrum at the highest-posterior retained sample (`theta_map_sample`) and compute credible intervals from posterior samples to compare with the observed data. This representative curve is not labelled as a maximum-likelihood fit.

In [ ]:
#get diagnostics along with the spectrum
gnostics = 0
# Now run the model again to get your model spectrum and process to make it look like the data

trimspec, photspec, tauspec,cfunc = test_module.modelspec(theta_map_sample,re_params,runargs,gnostics)
wave,topspec=proc_spec(inputspec=trimspec, theta=params_instance, re_params=re_params, args_instance=args_instance, do_scales=args_instance.do_scales, do_shift=args_instance.do_shift)


In [ ]:
# Now grab 500 random draws from the posterior
pltspec = np.zeros((500,runargs.obspec[0,:].size))
samp= np.empty(ndim)
samples = flatendchain
sid = np.zeros(500)
for i in range (0,500):
    sid[i]= np.random.randint(0,high = len(samples))
    samp = samples[int(sid[i]),:]
    
    trimspec, photspec, tauspec,cfunc = test_module.modelspec(samp,re_params,runargs,gnostics)
    pltspec[i,:] = proc_spec(inputspec=trimspec, theta=params_instance, re_params=re_params, args_instance=args_instance, do_scales=args_instance.do_scales, do_shift=args_instance.do_shift)[1]

# get the intervals for the distribution of model spectra
specdist = np.empty([runargs.obspec[0].size,5])
for i in range(0,runargs.obspec[0].size):
    junk = pltspec[:,i]
    junk2 = np.percentile(junk, [2.4,16, 50, 84,97.6],axis=0)
    junk3 = np.array(junk2)
    specdist[i,:] = junk3[:]


In [ ]:
# plot the spectra
plt.rc('font',family='Times New Roman')
fig=plt.figure(dpi=120)


d1, = plt.plot(runargs.obspec[0,:],runargs.obspec[1,:],'k-',label = "data")
t1, = plt.plot(runargs.obspec[0,:],topspec,'g-',linewidth=1, label = runname+" highest-posterior sample")

r1, = plt.plot(runargs.obspec[0],specdist[:,2],'y-',linewidth=0.5, label = "median")
#plt.fill_between(obspec[0],specdist[:,0],specdist[:,4],facecolor='red',alpha=0.2)
plt.fill_between(runargs.obspec[0],specdist[:,1],specdist[:,3],facecolor='red',alpha=0.5)
#plt.fill_between(obspec[0,:],obspec[1,:]-obspec[2,:],obspec[1,:]+obspec[2,:],facecolor='red',alpha=0.2)


plt.legend(handles=[d1,t1,r1])


plt.ylabel(r'$ F_{\lambda}$ / $Wm^{-2} \mu m^{-1}$')
plt.xlabel('Wavelength / $\mu m$')
#plt.savefig(runname+"_SPAG_SPEC.png",format='png', dpi=320))